1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split 
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

2. Load Data

In [2]:
# dataset path
DATA_PATH = "../../annotations"
os.listdir(DATA_PATH)

['lisa_dataset_split.csv', 'new_annotations.csv']

In [3]:
# Load the precomputed split.
# Expects 'lisa_dataset_split.csv' to be located under DATA_PATH.
df = pd.read_csv(os.path.join(DATA_PATH,'new_annotations.csv'), sep=';')

In [4]:
# Preview first 5 rows
df.head()

,image_id,label,isNight,split
0,../../datasets/processed_images\train\img_0.jpg,1,0,train
1,../../datasets/processed_images\train\img_1.jpg,1,0,train
2,../../datasets/processed_images\train\img_2.jpg,1,0,train
3,../../datasets/processed_images\train\img_3.jpg,1,0,train
4,../../datasets/processed_images\train\img_4.jpg,1,0,train


In [5]:
df.shape

(51826, 4)

3. Build ANN-ready datasets

In [6]:
CSV_PATH   = "../../annotations/new_annotations.csv"
IMG_SIZE   = 50
NUM_CLASSES = 3  # stop / warning / go mapped to 0..2

# Auto-detect separator (handles ';' from previous exports)
df = pd.read_csv(CSV_PATH, sep=None, engine="python")

# Quick sanity checks on splits and label balance
print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

# Labels come as 1/2/3 -> remap to 0/1/2
df["label"] = df["label"].astype(int) - 1
assert df["label"].between(0, NUM_CLASSES-1).all(), "Found labels outside 0..2"

def build_Xy(df_part):
    """
    Build design matrix and one-hot targets from a dataframe slice.

    - Reads each image path, resizes to IMG_SIZE x IMG_SIZE, normalizes to [0,1]
    - Flattens the image to a single vector
    - Appends the binary context feature `isNight` as an extra column
    - One-hot encodes labels to shape (N, NUM_CLASSES)
    """
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label"])
        is_night = int(row["isNight"])

        # Skip missing or unreadable images
        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        # Resize and normalize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0
        
        # Collect features and target
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    # Flatten images: (N, H, W, C) -> (N, H*W*C)
    X_images = np.array(X_images).reshape(len(X_images), -1)
    # Context feature as (N, 1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    # Concatenate image vector with isNight -> (N, H*W*C + 1)
    X_combined = np.hstack([X_images, X_isNight])

    # One-hot encode targets
    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y

# Build split-specific datasets from the 'split' column
X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "val"])
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)

Splits: {'train': 30707, 'test': 14433, 'val': 6686}
Labels per split:
 label      1    2      3
split                   
test    7229  499   6705
train  14272  800  15635
val     2681  256   3749


100%|██████████| 14433/14433 [01:05<00:00, 221.00it/s]


Shapes: 
 X_train: (30707, 7501)  y_train: (30707, 3) 
 X_val:   (6686, 7501)  y_val:   (6686, 3) 
 X_test:  (14433, 7501)  y_test:  (14433, 3)


4. Baseline ANN classifier

In [7]:
# Feed-forward baseline on flattened image vectors (+1 context feat: isNight)
# Input dim comes from the prepared training matrix: X_train.shape[1] = IMG_SIZE*IMG_SIZE*3 + 1
model = Sequential([
    Input(shape=(X_train.shape[1],)),           # dynamic input size from the dataset
    Dense(128, activation='relu'),              # hidden layer 1
    Dense(64, activation='relu'),               # hidden layer 2
    Dense(NUM_CLASSES, activation='softmax')    # 3-way classification: stop/warning/go
])

# Standard multi-class setup: Adam optimizer + categorical cross-entropy + accuracy metric
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Model overview (layers, output shapes, params)
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               960256    
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 3)                 195       
                                                                 
Total params: 968707 (3.70 MB)
Trainable params: 968707 (3.70 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


5. Training loop with early stopping

In [8]:
# Stop training when validation loss hasn’t improved for 5 epochs,
# and roll back to the best weights observed during training.
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Fit baseline ANN
# - shuffle=True to decorrelate batches
# - validation_data used for early stopping & monitoring generalization
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True
)

Epoch 1/50


480/480 [==============================] - 4s 7ms/step - loss: 0.0515 - accuracy: 0.9855 - val_loss: 0.0440 - val_accuracy: 0.9898
Epoch 2/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0182 - accuracy: 0.9952 - val_loss: 0.0278 - val_accuracy: 0.9952
Epoch 3/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0142 - accuracy: 0.9964 - val_loss: 0.3987 - val_accuracy: 0.9222
Epoch 4/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0141 - accuracy: 0.9964 - val_loss: 0.0788 - val_accuracy: 0.9629
Epoch 5/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0106 - accuracy: 0.9972 - val_loss: 0.0622 - val_accuracy: 0.9647
Epoch 6/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0059 - accuracy: 0.9985 - val_loss: 0.2921 - val_accuracy: 0.9420
Epoch 7/50
480/480 [==============================] - 3s 7ms/step - loss: 0.0111 - accuracy: 0.9973 - val_loss: 0.0436 - val_accuracy: 0.9864


6. Evaluation

In [9]:
# Evaluate on the never-seen test split to estimate generalization.
# Use verbose=0 to keep output clean.
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")


Test loss: 0.0101 | Test acc: 0.9982


In [10]:

# Predict class probabilities on the test set (batched for speed/memory).
y_pred = model.predict(X_test, batch_size=512)

# Convert one-hot (or probas) to hard class indices.
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

# Per-class precision/recall/F1 + macro/micro averages.
print(classification_report(y_true_labels, y_pred_labels, digits=4))

# Raw confusion matrix counts: rows = true, columns = predicted.
cm = confusion_matrix(y_true_labels, y_pred_labels)
print("Confusion Matrix:\n", cm)


29/29 [==============================] - 0s 4ms/step
              precision    recall  f1-score   support

           0     0.9999    0.9978    0.9988      7229
           1     0.9840    0.9880    0.9860       499
           2     0.9975    0.9994    0.9984      6705

    accuracy                         0.9982     14433
   macro avg     0.9938    0.9951    0.9944     14433
weighted avg     0.9982    0.9982    0.9982     14433

Confusion Matrix:
 [[7213    5   11]
 [   0  493    6]
 [   1    3 6701]]


7. Save the trained model

In [ ]:
import os

MODEL_DIR = "../../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "traffic_light_ann.keras")

model.save(MODEL_PATH)